# 과제 3. 새 자료에서 DDPM 학습과 표집
*Assignment 3. DDPM on a new dataset: noise-prediction training and sampling*

**9주차 실습에서 출발합니다.** 망 · 잡음 요리법 · 학습 설정은 실습과 같고, 자료가 다릅니다. 실습의 네 군집 대신 **고리 위 여섯 군집**이고, 그중 하나는 점이 4% 뿐인 **드문 군집**입니다.
*This assignment starts from the Week 9 lab. The network, the noise recipe and the training setup are the lab's; the data are new: six clusters on a ring instead of four, and one of them is a rare cluster holding only 4% of the points.*

| 단계 / Step | 하는 일 / What you do | 코드 / Code | 배점 / Points |
|---|---|---|---|
| Q1 | 새 자료를 잡음으로 녹이기 / Melt the new data into noise | Code 1 | 2 |
| Q2 | 잡음 예측으로 학습하기 / Train by predicting the noise | Code 2 | 3 |
| Q3 | 잡음에서 표본 만들기 / Make samples from noise | Code 3 | 3 |
| Q4 | 드문 군집과 베끼기 점검 / The rare cluster and a copy check | 없음 / none | 3 |
| | 코드 세 칸 모두 통과 / all three code checks pass | | 1 |

**쓰는 법 / How to use it**
- 파일 > Drive에 사본 저장 뒤, **값이나 코드를 바꿨으면 언제나 런타임 > 모두 실행** 하세요. CPU 런타임이면 충분하고 전체가 1분 안팎입니다.
  *Save a copy in Drive, then after any change use **Runtime > Run all**. A CPU runtime is enough; the whole notebook takes about a minute.*
- 채울 것은 `None` 이 있는 **코드 세 줄**과 마지막 영수증 칸뿐입니다. 코드 한 줄이 틀려도 다음 문항은 그대로 열립니다. 분석은 늘 과목의 기준 구현으로 돌고, 여러분의 줄은 따로 점검합니다. 틀린 줄은 코드 완료 1점에만 영향을 줍니다.
  *You fill in only the **three code lines** marked `None` and the receipt fields. A wrong line never blocks a later question: the analysis always runs on the course's reference version, and your lines are checked separately. A wrong line affects only the 1 coding point.*
- 코드 칸에 빨간 `SyntaxError` 가 뜨면 그 줄을 `None` 으로 되돌리고 다시 모두 실행하세요. 문법 오류는 그 칸이 실행되지 않아 모두 실행이 거기서 멈춥니다.
  *If a code cell shows a red `SyntaxError`, set that line back to `None` and run all again. A syntax error stops Run all at that cell.*
- 각 문항의 출력 이름(예: `Q2_READ checkpoints`)을 읽고 한두 문장으로 답하세요. 답에 필요한 숫자는 모두 마지막 영수증에 모입니다.
  *Read the named output for each question (for example `Q2_READ checkpoints`) and answer in one or two sentences. Every number you need is collected in the receipt at the end.*

**제출 / Submission.** 두 파일. (1) Q1–Q4 답과 **영수증 블록**을 글자로 붙인 PDF, (2) 출력을 저장한 이 IPYNB. 마감 11월 27일(금) 23:59.
*Two files: (1) a PDF with your answers to Q1–Q4 and the **receipt block** pasted as text, (2) this IPYNB with its outputs saved. Due Friday 27 November, 23:59.*

In [ ]:
#@title 준비 / Setup. 먼저 실행하세요 / Run this first { display-mode: "form" }
import math, time, copy, json, hashlib, platform, datetime
import numpy as np, pandas as pd
import torch, torch.nn as nn
import matplotlib.pyplot as plt

torch.set_num_threads(2)                     # 실습과 같음. 같은 컴퓨터에서 다시 돌리면 같은 숫자 / as in the lab: a re-run on the same computer gives the same numbers
GREEN, GOLD, RED, GRAY = "#406551", "#C9971C", "#B04A3A", "#9AA5A0"
NOTEBOOK = "A3-DGM v4"

# 잡음 요리법. 실습과 같다 / The noise recipe, as in the lab
T = 100
betas = torch.cat([torch.zeros(1), torch.linspace(0.0008, 0.08, T)])
alphas = 1 - betas
abar = torch.cumprod(alphas, 0)

def t_features(t):
    """걸음 t 를 사인 · 코사인 16개로 편다. 실습과 같다 / spread step t into 16 sines and cosines, as in the lab."""
    tt = (t.float()[:, None] - 1) / T
    freqs = (2 ** torch.arange(8).float())[None, :]
    return torch.cat([torch.sin(tt * freqs * math.pi), torch.cos(tt * freqs * math.pi)], 1)

class Net(nn.Module):
    """잡음 예측기 ε_θ(x_t, t). 기본은 실습과 같은 은닉 64 × 2 / the noise predictor; by default the lab's hidden 64 × 2."""
    def __init__(self, hidden=64, depth=2):
        super().__init__()
        layers, d = [], 18
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.SiLU()]
            d = hidden
        self.net = nn.Sequential(*layers, nn.Linear(d, 2))
    def forward(self, x, t):
        return self.net(torch.cat([x, t_features(t)], 1))

# 기준 구현. 아래 분석은 늘 이 셋을 쓴다. 여러분의 코드 세 줄은 따로 점검한다
# Reference versions. The analysis below always uses these three; your three code lines are checked separately
def _jump(x0, eps, abar_t):
    return torch.sqrt(abar_t) * x0 + torch.sqrt(1 - abar_t) * eps
def _noise_loss(eps_hat, eps, x0):
    return (eps_hat - eps).square().mean()
def _ddpm_mean(x_t, eps_hat, beta_t, alpha_t, abar_t):
    return (x_t - beta_t / torch.sqrt(1 - abar_t) * eps_hat) / torch.sqrt(alpha_t)

def train(data, updates=2000, hidden=64, depth=2, seed=29, ckpts=(150, 600, 2000)):
    """실습 §3 과 같은 학습: 배치 512, 걸음 t 를 무작위로, Adam 1e-3 / the lab's training: batch 512, random t, Adam 1e-3."""
    gen = torch.Generator().manual_seed(seed); torch.manual_seed(seed)
    model = Net(hidden, depth); opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses, ck = [], {}
    for step in range(1, updates + 1):
        idx = torch.randint(0, len(data), (512,), generator=gen); x0 = data[idx]
        t = torch.randint(1, T + 1, (len(x0),), generator=gen)
        eps = torch.randn(x0.shape, generator=gen)
        loss = _noise_loss(model(_jump(x0, eps, abar[t][:, None]), t), eps, x0)
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
        if step in ckpts:
            ck[step] = copy.deepcopy(model.state_dict())
    return model, losses, ck

@torch.no_grad()
def sample(model, n, gen, fresh=True, x_T=None):
    """잡음에서 T 걸음을 거꾸로. fresh=False 면 새 잡음 σ_t z 를 더하지 않는다 / walk T steps back from noise; fresh=False adds no new noise."""
    x = torch.randn(n, 2, generator=gen) if x_T is None else x_T.clone()
    for t in range(T, 0, -1):
        x = _ddpm_mean(x, model(x, torch.full((len(x),), t)), betas[t], alphas[t], abar[t])
        if fresh and t > 1:
            x = x + torch.sqrt(betas[t]) * torch.randn(x.shape, generator=gen)
    return x

# 새 자료: 반지름 2.5 고리 위 여섯 군집. 0 번(위쪽)이 드문 군집 / new data: six clusters on a ring of radius 2.5; cluster 0 (top) is rare
ANGLES = torch.tensor([90., 150., 210., 270., 330., 30.]) * math.pi / 180
CENTERS = torch.stack([2.5 * torch.cos(ANGLES), 2.5 * torch.sin(ANGLES)], 1)
WEIGHTS = torch.tensor([0.04] + [0.96 / 5] * 5)
SPREAD = 0.2

def make_data(n, gen):
    ids = torch.multinomial(WEIGHTS, n, replacement=True, generator=gen)
    return CENTERS[ids] + torch.randn(n, 2, generator=gen) * SPREAD, ids

def describe(pts):
    """중심 1.0 안의 비율, 드문 군집의 비율, 군집 퍼짐 cluster spread, 군집별 점 수(중심 1.0 안).
    군집 퍼짐은 9주차 덱 「As the Loss Falls, Do the Samples Match the Data?」 의 spread: 모든 표본을 가장 가까운 중심으로 나누고,
    군집마다 두 좌표의 표준편차를 평균한 뒤 군집끼리 평균한다. 거리로 거르지 않으므로 군집 사이로 흘린 표본도 들어간다.
    Share within 1.0 of a center, the rare cluster's share, the cluster spread, and points per cluster (within 1.0).
    The cluster spread is the spread on the Week 9 slide: every sample goes to its nearest center; per cluster, average the standard
    deviations of the two coordinates, then average over clusters. No distance filter, so samples spilled between clusters count."""
    d, which = torch.cdist(pts, CENTERS).min(1)
    near = d < 1.0
    sp = [pts[which == k].std(0, correction=0).mean().item() for k in range(6) if (which == k).sum() > 5]   # 다섯 점 이하인 군집은 재지 않음 / clusters with five points or fewer are not measured
    return {"near_%": 100 * near.float().mean().item(), "rare_%": 100 * (near & (which == 0)).float().mean().item(),
            "cluster_spread": float(np.mean(sp)), "counts": [int((near & (which == k)).sum()) for k in range(6)]}

def within(pts, ref, r=0.05):
    """pts 가운데 ref 의 어느 점에서든 r 안에 든 비율(%) / % of pts within r of some point of ref."""
    return 100 * (torch.cdist(pts, ref).min(1).values < r).float().mean().item()

RESULTS, CHECKS = {}, {}

def table(name, df):
    RESULTS[name] = df.to_json(orient="split", double_precision=6)
    print(name); print(df.round(3).to_string(index=False)); print()

def check(n, run, expected, wrong=()):
    """점검은 멈추지 않는다 / a check never stops the notebook."""
    try:
        got = run()
        if got is None:
            CHECKS[n] = False
            print(f"Code {n}: 아직 비어 있음 / still blank (None). 다음 문항은 그대로 진행됩니다 / later questions still run."); return
        got = torch.as_tensor(got, dtype=torch.float32)
        if got.shape == expected.shape and torch.allclose(got, expected, atol=1e-3):
            CHECKS[n] = True; print(f"Code {n}: PASSED"); return
        CHECKS[n] = False
        for val, msg in wrong:
            if got.shape == val.shape and torch.allclose(got, val, atol=1e-3):
                print(f"Code {n}: NOT PASSED. {msg}"); return
        print(f"Code {n}: NOT PASSED. 얻은 값 / got {got.numpy().round(4).tolist()}, 기대 / expected {expected.numpy().round(4).tolist()}")
    except Exception as e:
        CHECKS[n] = False
        print(f"Code {n}: NOT PASSED ({type(e).__name__}: {str(e)[:120]}). 다음 문항은 그대로 진행됩니다 / later questions still run.")

print("준비 완료 / Setup ready  |  torch", torch.__version__, "| threads", torch.get_num_threads())

In [ ]:
#@title 자료 / Data. 고리 위 여섯 군집, 하나는 드문 군집 / Six clusters on a ring, one of them rare { display-mode: "form" }
data, data_ids = make_data(2000, torch.Generator().manual_seed(29))     # 학습 자료 2,000점 / 2,000 training points
held, held_ids = make_data(2000, torch.Generator().manual_seed(30))     # 보류한 진짜 자료. 학습에 쓰지 않는다 / held-out real data, never used for training
DATA_HASH = hashlib.sha256(data.numpy().tobytes()).hexdigest()[:16]

info = describe(data)
table("DATA", pd.DataFrame([{"points": len(data), "rare_cluster_points": int((data_ids == 0).sum()),
                             "rare_%": 100 * (data_ids == 0).float().mean().item(), "cluster_spread": info["cluster_spread"]}]))
fig, ax = plt.subplots(figsize=(3.4, 3.4))
ax.scatter(data[:, 0], data[:, 1], s=2, color=GREEN, alpha=0.5)
ax.scatter(CENTERS[1:, 0], CENTERS[1:, 1], marker="+", s=70, color=GOLD, linewidths=1.6)
ax.scatter(CENTERS[:1, 0], CENTERS[:1, 1], marker="+", s=110, color=RED, linewidths=2.0)
ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title="Training data (red +: the rare cluster)")
ax.set_aspect("equal"); plt.tight_layout(); plt.show()
# 기대값 / Expected   rare_cluster_points 84, rare_% 4.2, cluster_spread 0.206

---
## Q1 · 새 자료를 잡음으로 녹이기 / Melt the new data into noise  `2점 / 2 points`
9주차 실습 §1 · §2 와 같은 요리법으로 새 자료를 녹입니다. 먼저 Code 1(한 줄 건너뛰기)을 채우세요. 「overall spread」(전체 퍼짐) 는 9주차 실습 §1 의 「퍼짐」 처럼 점 전체가 얼마나 흩어졌는지입니다. 여기서는 두 좌표 분산의 평균의 제곱근으로 잽니다(식과 같은 정의).

*Start with the fixed half of a diffusion model. Read Q1_READ spread and the figure; Code 1 is the one-line jump to any step. The overall spread is the spread of all points together: the square root of the average of the two coordinate variances.*

(a) For this data the overall spread ends near 1 at t = 100. Why does the recipe pull the points toward N(0, I), and what does the signal coefficient √ᾱ_t, about 0.125 at t = 100, say about how much of the data is left? [1]
(b) Nothing in Q1 used a network. Which part of a diffusion model is fixed by the recipe, and which part has to be learned? [1]

In [ ]:
#@title Code 1 ▶ 어느 걸음이든 한 줄로 / Jump to any step in one line { display-mode: "form" }
# 다음 이름을 씁니다 / Names used below
#   x0      : 깨끗한 점들, 크기 (점 수, 2) / the clean points, shape (number of points, 2)
#   eps     : 점마다 뽑은 잡음 ε, x0 와 같은 크기 / the noise ε drawn for each point
#   abar_t  : 점마다의 ᾱ_t, 크기 (점 수, 1) / each point's ᾱ_t, shape (number of points, 1)
#   x_t     : 잡음이 섞인 점들   <- 빈칸 / the noisy points   <- the blank
# 참고 / Reference   torch.sqrt(a) 는 칸마다 제곱근 / the square root of every entry
def jump(x0, eps, abar_t):
    """x_0 에서 걸음 t 의 x_t 로 한 줄에 간다. 9주차 슬라이드의 한 줄 공식 / go from x_0 to x_t in one line: the Week 9 one-line formula."""
    # TODO 1. 9주차 슬라이드 「Jump to Any Step in One Line」 의 식을 그대로 쓰세요. 한 줄입니다.
    # TODO 1. Write the formula on the Week 9 slide "Jump to Any Step in One Line" as it is. One line.
    x_t = None
    return x_t

check(1, lambda: jump(torch.tensor([[-2.0, 2.0]]), torch.tensor([[0.5, -0.5]]), torch.tensor([[0.25]])),
      torch.tensor([[-0.5670, 0.5670]]),
      [(torch.tensor([[-0.125, 0.125]]), "ᾱ_t 에 제곱근이 빠졌습니다. 계수는 √ᾱ_t 와 √(1 − ᾱ_t) / the square roots are missing: √ᾱ_t and √(1 − ᾱ_t)."),
       (torch.tensor([[-0.6250, 0.6250]]), "잡음 쪽 계수에만 제곱근이 빠졌습니다 / the square root is missing on the noise coefficient only."),
       (torch.tensor([[-1.4821, 1.4821]]), "두 계수가 서로 바뀌었습니다. 신호에 √ᾱ_t, 잡음에 √(1 − ᾱ_t) / the two coefficients are swapped.")])

In [ ]:
#@title Q1 제공 분석 / Provided analysis { display-mode: "form" }
gen = torch.Generator().manual_seed(1)
var0 = data.var(0).mean().item()
rows, snaps = [], {}
for t in (0, 10, 25, 50, 100):
    pts = data if t == 0 else _jump(data, torch.randn(data.shape, generator=gen), abar[t].expand(len(data), 1))
    snaps[t] = pts
    # overall spread = √(두 좌표 분산의 평균). 점과 식이 같은 정의 / √(average of the two coordinate variances), the same definition for the points and the formula
    rows.append({"t": t, "overall_spread_of_points": pts.var(0).mean().sqrt().item(),
                 "overall_spread_from_formula": math.sqrt(abar[t] * var0 + (1 - abar[t])), "signal_coefficient_sqrt_abar": abar[t].sqrt().item()})
table("Q1_READ spread", pd.DataFrame(rows))
fig, axs = plt.subplots(1, 5, figsize=(11, 2.5))
for ax, t in zip(axs, (0, 10, 25, 50, 100)):
    ax.scatter(snaps[t][:, 0], snaps[t][:, 1], s=2, color=GREEN, alpha=0.5)
    ax.set(xlim=(-5, 5), ylim=(-5, 5), xticks=[-4, 0, 4], yticks=[-4, 0, 4], title=f"t = {t}"); ax.set_aspect("equal")
plt.suptitle("The six clusters melting into noise", fontsize=10); plt.tight_layout(); plt.show()

---
## Q2 · 잡음 예측으로 학습하기 / Train by predicting the noise  `3점 / 3 points`
실습 §3 과 같은 망과 학습(2,000회)입니다. 학습의 한가운데 줄, 잡음 예측의 손실을 Code 2 로 직접 씁니다. 「cluster spread」(군집 퍼짐) 는 9주차 덱 「As the Loss Falls, Do the Samples Match the Data?」 의 spread 입니다. 모든 표본을 가장 가까운 중심으로 나누고, 군집마다 두 좌표의 표준편차를 평균한 뒤 군집끼리 평균합니다. 군집 사이로 흘린 표본도 들어갑니다.

*Now the learned half. Code 2 is the line at the heart of training, the noise-prediction loss. Read Q2_READ loss and Q2_READ checkpoints; the last row of the checkpoint table is the real data. The cluster spread is the spread on the Week 9 slide "As the Loss Falls, Do the Samples Match the Data?": every sample goes to its nearest center, the standard deviations of each cluster's two coordinates are averaged, and then averaged over the clusters. Samples spilled between clusters count too.*

(a) Even with a well-trained network, why is the average noise-prediction loss not necessarily zero? Explain using what the network observes: x_t and t. [1]
(b) As the loss falls, how do the share of samples near a center, the cluster spread and the rare cluster's share change, and which improves first? Compare with the real-data row. [2]

In [ ]:
#@title Code 2 ▶ 잡음 예측의 손실 / The noise-prediction loss { display-mode: "form" }
# 다음 이름을 씁니다 / Names used below
#   eps_hat : 신경망이 예측한 잡음 ε̂, 크기 (점 수, 2) / the network's predicted noise
#   eps     : 실제로 섞은 잡음 ε / the noise that was actually mixed in
#   x0      : 깨끗한 점 / the clean points
#   loss    : 평균 제곱 오차   <- 빈칸 / the mean squared error   <- the blank
# 참고 / Reference   (a - b).square().mean() 은 칸마다 차를 제곱해 평균 / squares each difference and averages
def noise_loss(eps_hat, eps, x0):
    """망이 맞혀야 할 것과 망의 답의 평균 제곱 오차. 9주차: 망은 무엇을 예측하나 / mean squared error between the network's answer and what it must predict (Week 9)."""
    # TODO 2. 망이 맞혀야 할 것(9주차 「The Network Learns to Predict the Noise」)과 망의 답 eps_hat 의 평균 제곱 오차를 쓰세요. 한 줄입니다.
    # TODO 2. Write the mean squared error between eps_hat and what the network must predict (Week 9, "The Network Learns to Predict the Noise"). One line.
    loss = None
    return loss

check(2, lambda: noise_loss(torch.tensor([[0.1, 0.5]]), torch.tensor([[0.3, -0.2]]), torch.tensor([[2.0, 2.0]])),
      torch.tensor(0.265),
      [(torch.tensor(2.93), "x0 와 견줬습니다. 망이 예측하는 것은 섞은 잡음 ε 입니다 / you compared with x0; the network predicts the noise ε."),
       (torch.tensor(0.53), "평균 대신 합을 했습니다 / you summed instead of averaging."),
       (torch.tensor(0.25), "제곱이 빠졌습니다 / the square is missing."),
       (torch.tensor(0.45), "제곱 대신 절댓값을 썼습니다 / you used absolute values instead of squares.")])

In [ ]:
#@title Q2 제공 분석 / Provided analysis. 학습 10초 안팎 / training takes about 10 s { display-mode: "form" }
t0 = time.time()
model, losses, ckpt = train(data)
loss_first, loss_last = float(np.mean(losses[:100])), float(np.mean(losses[-100:]))
table("Q2_READ loss", pd.DataFrame([{"mean_loss_first_100": loss_first, "mean_loss_last_100": loss_last, "seconds": time.time() - t0}]))
plt.figure(figsize=(5, 2.6))
plt.plot(losses, color=GRAY, lw=0.5, alpha=0.8)
k = 50; plt.plot(np.arange(k, len(losses) + 1), np.convolve(losses, np.ones(k) / k, mode="valid"), color=GREEN, lw=1.5)
for s in (150, 600, 2000): plt.axvline(s, color=GOLD, lw=0.8, ls=":")
plt.xlabel("Adam updates"); plt.ylabel("noise MSE"); plt.ylim(0.25, 1.05); plt.title("Training loss (gold: checkpoints)", fontsize=10)
plt.tight_layout(); plt.show()

rows, fig = [], plt.figure(figsize=(11, 2.8))
for i, s in enumerate((150, 600, 2000)):
    net = Net(); net.load_state_dict(ckpt[s])
    pts = sample(net, 2000, torch.Generator().manual_seed(1000 + s))
    d = describe(pts); rows.append({"updates": s, "near_%": d["near_%"], "rare_%": d["rare_%"], "cluster_spread": d["cluster_spread"]})
    ax = fig.add_subplot(1, 4, i + 1); ax.scatter(pts[:, 0], pts[:, 1], s=2, color=GREEN, alpha=0.45)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title=f"{s} updates"); ax.set_aspect("equal")
d = describe(data); rows.append({"updates": "real data", "near_%": d["near_%"], "rare_%": d["rare_%"], "cluster_spread": d["cluster_spread"]})
ax = fig.add_subplot(1, 4, 4); ax.scatter(data[:, 0], data[:, 1], s=2, color=GRAY, alpha=0.45)
ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title="real data"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()
table("Q2_READ checkpoints", pd.DataFrame(rows))

---
## Q3 · 잡음에서 표본 만들기 / Make samples from noise  `3점 / 3 points`
실습 §4 의 역방향 한 걸음입니다. Code 3 에 평균 부분을 쓰세요. 아래 분석은 학습을 마친 망으로, 역방향 걸음마다(마지막 걸음 제외) 새 잡음 σ_t z 를 더한 표집과 그 새 잡음을 뺀 표집을 견줍니다.

*A trained network still has to walk back from noise. Code 3 is the mean of one reverse step. Read Q3_READ fresh noise for (a) and Q3_READ same start for (b). Dropping the fresh noise is not DDIM; DDIM has its own update rule.*

(a) What happens to the cluster spread and to the rare cluster when the fresh noise is dropped? What role does fresh Gaussian noise play in DDPM reverse sampling, except at the final step? [1.5]
(b) Starting twice from the same noise x_T gave different samples. Why? Each sample also took 100 network passes: what does this cost mean next to a VAE or GAN from earlier weeks? [1.5]

In [ ]:
#@title Code 3 ▶ 역방향 한 걸음의 평균 / The mean of one reverse step { display-mode: "form" }
# 다음 이름을 씁니다 / Names used below
#   x_t, eps_hat : 지금 걸음의 점들과 예측한 잡음 / the points at this step and the predicted noise
#   beta_t, alpha_t, abar_t : 이 걸음의 β_t, α_t = 1 − β_t, 누적 ᾱ_t / this step's β_t, α_t, and the running ᾱ_t
#   mean    : 새 잡음을 더하기 전의 x_{t−1}   <- 빈칸 / x_{t−1} before the new noise   <- the blank
def ddpm_mean(x_t, eps_hat, beta_t, alpha_t, abar_t):
    """예측한 잡음의 일부를 빼고 줄였던 신호를 되돌린다. 9주차 역방향 한 걸음 / remove part of the predicted noise and undo the shrink (Week 9)."""
    # TODO 3. 9주차 슬라이드 「One DDPM Reverse Step」 의 평균 부분을 쓰세요. 한 줄입니다.
    # TODO 3. Write the mean part of the Week 9 slide "One DDPM Reverse Step". One line.
    mean = None
    return mean

check(3, lambda: ddpm_mean(torch.tensor([[1.0]]), torch.tensor([[0.6]]), torch.tensor(0.01004), torch.tensor(0.98996), torch.tensor(0.07859)),
      torch.tensor([[0.9988]]),
      [(torch.tensor([[0.9937]]), "√α_t 로 나누지 않았습니다 / you did not divide by √α_t."),
       (torch.tensor([[3.5447]]), "α_t 대신 ᾱ_t 로 나눴습니다 / you divided by ᾱ_t instead of α_t."),
       (torch.tensor([[0.9446]]), "ε̂ 의 계수에 √β_t 를 썼습니다. √β_t 는 새 잡음의 크기 / you used √β_t on ε̂; that is the size of the new noise.")])

In [ ]:
#@title Q3 제공 분석 / Provided analysis { display-mode: "form" }
with_noise = sample(model, 2000, torch.Generator().manual_seed(3000), fresh=True)
no_noise = sample(model, 2000, torch.Generator().manual_seed(3000), fresh=False)
x_T = torch.randn(600, 2, generator=torch.Generator().manual_seed(7))
first = sample(model, 600, torch.Generator().manual_seed(11), x_T=x_T)
second = sample(model, 600, torch.Generator().manual_seed(12), x_T=x_T)
same_xT_distance = (first - second).norm(dim=1).mean().item()
rows = []
for name, pts in (("with fresh noise", with_noise), ("without fresh noise", no_noise), ("real data", data)):
    d = describe(pts)
    rows.append({"sampling": name, "near_%": d["near_%"], "rare_%": d["rare_%"], "cluster_spread": d["cluster_spread"],
                 "smallest_cluster": min(d["counts"]), "largest_cluster": max(d["counts"])})
table("Q3_READ fresh noise", pd.DataFrame(rows))
table("Q3_READ same start", pd.DataFrame([{"mean_distance_between_two_runs_from_same_x_T": same_xT_distance, "network_passes_per_sample": T}]))
fig, axs = plt.subplots(1, 3, figsize=(8.4, 2.8))
for ax, (name, pts, col) in zip(axs, (("with fresh noise", with_noise, GREEN), ("without fresh noise", no_noise, RED), ("real data", data, GRAY))):
    ax.scatter(pts[:, 0], pts[:, 1], s=2, color=col, alpha=0.45)
    ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title=name); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

---
## Q4 · 드문 군집과 베끼기 점검 / The rare cluster and a copy check  `3점 / 3 points`
9주차의 마지막 두 질문입니다. 만든 표본을 믿을 수 있나, 드문 경우를 제대로 만드나. 코드는 쓰지 않습니다. 베끼기 점검은 망을 두 번 더 학습해 30초 안팎 걸립니다.

*Good-looking samples are not yet useful data. Read Q4_READ rare cluster for (a) and Q4_READ copy check for (b). The copy check is the Week 9 method: compare each sample's distance to its nearest training point with the same distance for held-out real data. For this check a larger network (hidden 128 × 3, the size used in Week 9) is trained for 4,000 updates on 8 training points and on 2,000.*

(a) Does the model make the rare cluster at about its real share? How much do the three sampling runs differ, and what does that say about judging a small share? [1.5]
(b) With 8 training points and with 2,000, how often does a sample land within 0.05 of a training point, compared with held-out real points? Which result shows a stronger sign of samples sitting too close to the training points, and why must you compare with held-out real data instead of reading the sample share alone? [1.5]

In [ ]:
#@title Q4 제공 분석 / Provided analysis. 30초 안팎 / about 30 s { display-mode: "form" }
rows = []
for run in (1, 2, 3):
    d = describe(sample(model, 2000, torch.Generator().manual_seed(4000 + run)))
    rows.append({"sampling_run": run, "rare_%": d["rare_%"], "rare_cluster_points_of_2000": d["counts"][0]})
rows.append({"sampling_run": "data", "rare_%": 100 * (data_ids == 0).float().mean().item(), "rare_cluster_points_of_2000": int((data_ids == 0).sum())})
table("Q4_READ rare cluster", pd.DataFrame(rows))

small = data[:8]                                                       # 학습점 8개 / 8 training points
rows, fig = [], plt.figure(figsize=(6.4, 3.0))
for i, (name, tr) in enumerate((("8 points", small), ("2,000 points", data))):
    m, _, _ = train(tr, updates=4000, hidden=128, depth=3, seed=29, ckpts=())
    pts = sample(m, 300, torch.Generator().manual_seed(5))
    rows.append({"training_set": name, "samples_within_0.05_of_a_training_point_%": within(pts, tr),
                 "held_out_real_within_0.05_of_a_training_point_%": within(held[:300], tr)})
    ax = fig.add_subplot(1, 2, i + 1)
    ax.scatter(pts[:, 0], pts[:, 1], s=4, color=GREEN, alpha=0.5, label="samples")
    if len(tr) <= 8: ax.scatter(tr[:, 0], tr[:, 1], s=40, facecolors="none", edgecolors=RED, linewidths=1.4, label="training points")
    ax.set(xlim=(-4, 4), ylim=(-4, 4), xticks=[-2, 0, 2], yticks=[-2, 0, 2], title=f"trained on {name}"); ax.set_aspect("equal")
    if i == 0: ax.legend(fontsize=7, loc="lower left")
plt.tight_layout(); plt.show()
table("Q4_READ copy check", pd.DataFrame(rows))

---
## 분석 영수증 / Analysis receipt
학번과 AI 사용을 적고 **런타임 > 모두 실행** 하세요. `=====` 줄 사이의 영수증 블록만 PDF 에 글자로 붙입니다. 그 아래 기계 기록은 IPYNB 에 남으므로 PDF 에 넣지 않습니다. READY 는 노트북 결과가 다 갖춰졌다는 뜻이고, 서술형 답과 이 영수증을 PDF 에 넣어야 제출이 끝납니다.
*Fill in your student ID and AI use, then **Runtime > Run all**. Paste only the receipt block between the `=====` lines into your PDF; the machine record below it stays in your IPYNB. READY means the notebook results are complete; you still need your written answers and this receipt in your PDF.*

In [ ]:
#@title Analysis Receipt / 이 블록을 PDF에 복사 / Copy this block into your PDF { display-mode: "form" }
student_id = ""  #@param {type:"string"}
ai_use = ""  #@param {type:"string"}

need = ["DATA", "Q1_READ spread", "Q2_READ loss", "Q2_READ checkpoints", "Q3_READ fresh noise", "Q3_READ same start", "Q4_READ rare cluster", "Q4_READ copy check"]
missing = [k for k in need if k not in RESULTS]
consistent = not missing and DATA_HASH == hashlib.sha256(data.numpy().tobytes()).hexdigest()[:16]
todo = []
if missing: todo.append("Runtime > Run all")
miss_code = [str(i) for i in (1, 2, 3) if not CHECKS.get(i, False)]
if miss_code: todo.append("Code " + ",".join(miss_code) + " not passed: coding point only / 코드 점수만 영향")
if not student_id.strip(): todo.append("student ID / 학번")
if not ai_use.strip(): todo.append("AI use / AI 사용란")
ready = consistent and not todo
receipt = {"notebook": NOTEBOOK, "student_id": student_id.strip(), "consistency": "OK" if consistent else "NOT OK",
           "submission": "READY" if ready else "NOT READY", "checks": {str(k): bool(v) for k, v in CHECKS.items()},
           "coding_point": 1 if all(CHECKS.get(i, False) for i in (1, 2, 3)) else 0, "data_sha256": DATA_HASH,
           "ai_use": ai_use.strip(), "results": RESULTS,
           "environment": {"python": platform.python_version(), "torch": torch.__version__, "numpy": np.__version__,
                           "device": "CPU", "gpu_available": bool(torch.cuda.is_available())},
           "run_at": datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}
receipt["record_hash"] = hashlib.sha256(json.dumps(receipt, sort_keys=True, default=str).encode()).hexdigest()

def _t(name):
    from io import StringIO
    return pd.read_json(StringIO(RESULTS[name]), orient="split") if name in RESULTS else None

L = [f"notebook         {NOTEBOOK}    student ID {student_id.strip() or '-'}",
     "consistency      " + ("OK" if consistent else "NOT OK: Runtime > Run all"),
     "submission       " + ("READY" if ready else "NOT READY")]
L += ["  to do          " + t for t in todo]
L.append("code checks      " + " ".join(f"{i}:{'PASS' if CHECKS.get(i, False) else '-'}" for i in (1, 2, 3)) + f"    coding point {receipt['coding_point']}")
q = _t("Q1_READ spread")
if q is not None:
    s = q.set_index("t")["overall_spread_of_points"]
    L.append(f"Q1 overall spread  t=0 {s[0]:.3f} | t=50 {s[50]:.3f} | t=100 {s[100]:.3f}")
q = _t("Q2_READ loss")
if q is not None:
    L.append(f"Q2 loss          first 100 {q.iloc[0]['mean_loss_first_100']:.3f} | last 100 {q.iloc[0]['mean_loss_last_100']:.3f}")
q = _t("Q2_READ checkpoints")
if q is not None:
    for _, r in q.iterrows():
        L.append(f"Q2 {str(r['updates']):>10s}    near {r['near_%']:.0f}% | rare {r['rare_%']:.1f}% | cluster spread {r['cluster_spread']:.3f}")
q = _t("Q3_READ fresh noise")
if q is not None:
    for _, r in q.iterrows():
        L.append(f"Q3 {r['sampling'][:20]:20s} rare {r['rare_%']:.1f}% | cluster spread {r['cluster_spread']:.3f} | clusters {int(r['smallest_cluster'])}-{int(r['largest_cluster'])}")
q = _t("Q3_READ same start")
if q is not None:
    L.append(f"Q3 same x_T      mean distance {q.iloc[0]['mean_distance_between_two_runs_from_same_x_T']:.2f} | passes per sample {int(q.iloc[0]['network_passes_per_sample'])}")
q = _t("Q4_READ rare cluster")
if q is not None:
    L.append("Q4 rare share    " + " | ".join(f"{('run ' + str(r['sampling_run'])) if str(r['sampling_run']) != 'data' else 'data'} {r['rare_%']:.1f}%" for _, r in q.iterrows()))
q = _t("Q4_READ copy check")
if q is not None:
    for _, r in q.iterrows():
        L.append(f"Q4 copy {r['training_set']:12s} samples {r.iloc[1]:.1f}% | held-out real {r.iloc[2]:.1f}%  (within 0.05)")
L.append(f"AI use           {ai_use.strip() or '(empty)'}")
L.append(f"environment      Python {platform.python_version()} | torch {torch.__version__} | computed on CPU (GPU available: {'yes' if torch.cuda.is_available() else 'no'})")
L.append(f"record hash      {receipt['record_hash'][:16]}    full record: next cell and your IPYNB")
print("===== ANALYSIS RECEIPT: copy from here into your PDF / 여기부터 PDF에 복사 =====")
for l in L: print(l)
print("===== END OF RECEIPT / 여기까지 =====")
if ready:
    print("READY 는 노트북 결과가 다 갖춰졌다는 뜻입니다. 서술형 답과 이 영수증을 PDF 에 넣어야 제출이 끝납니다.")
    print("READY means your notebook results are complete. You still need your written answers and this receipt in your PDF.")

In [ ]:
#@title 기계 기록 / Machine record. PDF에 복사하지 않습니다 / Do not copy into your PDF
# 채점자가 IPYNB 에서 확인하는 전체 기록입니다 / The grader checks this full record in your IPYNB.
print(json.dumps(receipt, ensure_ascii=False, indent=2, default=str))